In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
import numpy as np 



data_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
data_test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

In [2]:
X = data_train

In [3]:
X.head(20)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [4]:
X = data_train.drop(['PassengerId', 'Survived', 'Name','Ticket','Cabin'], axis = 1)
y = data_train.Survived
X = X.fillna({'Age' : X.Age.median()})
X = X.fillna({'Fare' : X.Fare.median()})
X = pd.get_dummies(X, dtype=int)
X = X.drop(['Sex_male'], axis = 1)

In [5]:
X.head()

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Embarked_C,Embarked_Q,Embarked_S
0,3,22.0,1,0,7.2500,0,0,0,1
1,1,38.0,1,0,71.2833,1,1,0,0
2,3,26.0,0,0,7.9250,1,0,0,1
3,1,35.0,1,0,53.1000,1,0,0,1
4,3,35.0,0,0,8.0500,0,0,0,1


In [6]:
X_test = data_test.drop(['PassengerId', 'Name','Ticket','Cabin'], axis = 1)
X_test = X_test.fillna({'Age' : X.Age.median()})
X_test = X_test.fillna({'Fare' : X.Fare.median()})
X_test = pd.get_dummies(X_test, dtype=int)
X_test = X_test.drop(['Sex_male'], axis = 1)

In [7]:
X, X_test = X.align(X_test, join='inner', axis=1)

In [8]:
clf = RandomForestClassifier(criterion='entropy', random_state=42, n_jobs=-1)
#parameters = {'n_estimators' : range(40,90,5), 'max_depth' : range(10,20,1), 'min_samples_split' : range(5,10,1)}
parameters = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': range(1,15,1),
    'min_samples_split': [2, 5, 10]
}
grid_search_CV_clf = GridSearchCV(clf, parameters, cv = 5, n_jobs=-1)
grid_search_CV_clf.fit(X,y)


GridSearchCV(cv=5,
             estimator=RandomForestClassifier(criterion='entropy', n_jobs=-1,
                                              random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': range(1, 15),
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [50, 100, 150, 200]})

In [9]:
grid_search_CV_clf.best_score_

np.float64(0.8383968363567886)

In [10]:
grid_search_CV_clf.best_params_


{'max_depth': 9, 'min_samples_split': 2, 'n_estimators': 200}

In [11]:
best_clf = grid_search_CV_clf.best_estimator_

In [12]:
feature_importances = best_clf.feature_importances_

In [13]:
feature_importances_df = pd.DataFrame({'features' :list(X), 'feature_importances' : feature_importances})

In [14]:
feature_importances_df.sort_values('feature_importances', ascending = False)

,features,feature_importances
5,Sex_female,0.304935
4,Fare,0.236074
1,Age,0.209175
0,Pclass,0.106176
2,SibSp,0.056802
3,Parch,0.043657
8,Embarked_S,0.017620
6,Embarked_C,0.016179
7,Embarked_Q,0.009382


In [15]:
predictions=grid_search_CV_clf.predict(X_test)

In [16]:
submission = pd.DataFrame({
    'PassengerId': data_test['PassengerId'],
    'Survived': predictions.astype(int)   
})
submission.to_csv('submission.csv', index=False)